In [53]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [54]:
load_dotenv()

True

In [55]:
subgraph_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [56]:
class SubState(TypedDict):

    input_text: str
    translated_text: str

In [57]:
def translate_text(state: SubState):
    prompt = f"""Translate the following text to Urdu,
     Keep it natural and clear do not add any extra content.
     Text : {state["input_text"]}""".strip()

    translated_text =  subgraph_llm.invoke(prompt).content

    return {"translated_text": translated_text}

In [58]:
subgraph_builder = StateGraph(SubState)

subgraph_builder.add_node("translate_text", translate_text)

subgraph_builder.add_edge(START,'translate_text')
subgraph_builder.add_edge('translate_text', END)

subgraph = subgraph_builder.compile()



In [59]:
class ParentState(TypedDict):
    question: str
    answer_eng: str
    answer_urdu: str


In [60]:
parent_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [61]:
def generate_answer(state: ParentState):
    answer = parent_llm.invoke(f"You are a helpful assistant, Answer clearly. \n {state['question']}").content
    return {'answer_eng':answer}

In [62]:
def translate_answer(state: ParentState):
    # Call the subgraph
    result = subgraph.invoke({'input_text':state['answer_eng']})
    return {"answer_urdu":result["translated_text"]}

In [63]:
parent_builder = StateGraph(ParentState)

parent_builder.add_node('generate_answer',generate_answer)
parent_builder.add_node('translate', translate_answer)

parent_builder.add_edge(START, 'generate_answer')
parent_builder.add_edge('generate_answer', 'translate')

parent_builder.add_edge('translate',END)

graph = parent_builder.compile()

In [64]:
graph.invoke({'question': "What is an AI Engineer"})

{'question': 'What is an AI Engineer',
 'answer_eng': "An **AI Engineer** is a specialized software engineer who focuses on designing, developing, and maintaining Artificial Intelligence (AI) systems and applications.\n\nTheir primary role is to bridge the gap between AI research (often done by data scientists or AI researchers) and practical, deployable software solutions. They take theoretical AI models and integrate them into real-world products and services, ensuring they are robust, scalable, and efficient.\n\nHere's a breakdown of what an AI Engineer does:\n\n1.  **Develop AI Applications:** They write code to build applications that incorporate AI functionalities, such as natural language processing, computer vision, recommendation systems, or predictive analytics.\n2.  **Integrate AI Models:** They take trained machine learning models (often developed by data scientists) and integrate them into existing software systems or build new systems around them.\n3.  **Data Engineering 